### Continued embedding reduction section

In [ ]:
# Run over the 3-8 kmeans cluster parameter of k and optimise the SILHOUTTE SCORE - search up, but essentially this is the quality of the clustering/separation:
from sklearn.metrics import silhouette_score
SAMPLE_SIZE = 100000 # This was upscaled from 50000 to 250000, but still didn't significantly increase silhoutte scores.
# NB SAMPLE_SIZE should be changed again if the runtime is sufficient, but this allows for fast comparisons with representativeness. 
# Random sampling:
idx = np.random.choice(len(embeddings), SAMPLE_SIZE, replace=False)
sample = embeddings[idx]


In [ ]:

## Added component to do away w silhoutte score problems: apply ONE dimensionality reduction (simplicity).
# This might allow us to reduce the curse of dimensionality (NOT to be confused with embedding collapse).

''' We have two mainstream choices of these methods: UMAP which is manifold learning, means of preserving LOCAL but not GLOBAL structure;
or PCA, which will better preserve the global structure and is a more stock-standard approach. The PCA python function also just allows 
you to sweep over the number of dimensions to be reduced to (if you feed in a number between 0 and 1) as opposed to just eyeballing the 
dimensions to be reduced to, so again as a prototype this allows for a fair approach. 
For reference, though, could also extend this approach: https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html
'''
import cuml
import cudf
from sklearn.decomposition import PCA

pca = PCA(n_components=0.95, random_state=67)
embeddings = pca.fit_transform(embeddings) # Dim reduction until we dip below 95% of variance being explained by the sum of the PCs

print(f"Dimensions retained at 95% variance: {pca.n_components_}")
print("we expect the shape of the reduced embeddingz to be (SAMPLE_SIZE, n_components). Shape is:", embeddings.shape)

print('...\nEmbeddings have now been fully cleaned/reduced. Moving on to clustering\n...')


## KMeansClustering 

NOTE that KMedoids was discussed, but because there is still a considerably large sample being taken and its computataional cost scales O(n^2) (computing between all points), we take KMeans (scales O(nk) (or oink), only comparing each vector's average) for prototyping. Maybe with the async optimisation KMedoids could be OK though.

In [ ]:


# sweep k=3 to k=8
scores = {}
for k in range(3, 9):
    print(f"Fitting k={k}...")
    km = MiniBatchKMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(embeddings_50d) # Fit to the REDUCED embeddingz
    score = silhouette_score(embeddings_50d, labels, metric="cosine")
    scores[k] = round(score, 3)
    print(f"  k={k} silhouette={score:.3f}") 

# find the optimal
best_k = max(scores, key=scores.get)
print(f"\nOptimal k={best_k} (score={scores[best_k]})")
print(f"Full scores OF EACH : {scores}")


print('Now optimal k has been found, we can fit the model to the FULL DATASET')
print('')
print('...')
print('')

# fit k-means to the full dataset with the optimal k
n_init = 10 # Depending on runtime on system-1 or other, this can be increased to take more random states of minibatches.
embeddings_50d_full = reducer.transform(embeddings) # Apply the same UMAP transformation to the FULL embeddingz
embeddings = embeddings_50d_full # For the rest of the code, we will work with ze reduced embeddingz. This has been shown in lit to still preserve local neighbourhoods and therefore should be fine for the clustering step.

# then fit k-means on the reduced version
kmeans = MiniBatchKMeans(n_clusters=best_k, random_state=42, n_init=n_init) #Fit to the swept best k
labels = kmeans.fit_predict(embeddings) # Fit the MiniBatch to embeddings
centroids = kmeans.cluster_centers_ # For similarity calculations later on ...
print(f"Fitted MiniBatchKMeans with k={best_k} to the FULL dataset.")

print(' ')
print('...')
print('')


## Final step - retrieving posts around cluster centroid

In [ ]:
## FINAL STEP - for input into Moltbook, we want to extract a certain number of samples (again this parameter can be optimised relative to runtime / what is needed for MiroFish)
# Take a 'n_posts' number of samples for each cluster. We could inspect manually, or put them into an LLM as with prior papers.

n_posts = 500 # Change if needed - for now just want to get this running and into a .txt file per persona.

from sklearn.metrics.pairwise import cosine_similarity # Our cosine similarity here works to calculate our distances from centroidz, and extract most representative post.
# In order to do this, we need to map it back onto the original 'texts':


for k in range(kmeans.n_clusters):
    print(f"\nCluster {k}:") # Each cluster being representative of a persona
    
    # get indices of all posts in this cluster
    cluster_mask = labels == k # Boolean 'masking' method - this allows us to map onto k < clusters
    cluster_indices = np.where(cluster_mask)[0]
    cluster_embeddings = embeddings[cluster_indices] 
    # from numpy docs - https://numpy.org/doc/stable/user/basics.indexing.html#boolean-array-indexing
    
    # compute cosine similarity of each post to its cluster centroid
    centroid = centroids[k].reshape(1, -1)
    similarities = cosine_similarity(cluster_embeddings, centroid).flatten() # simple 'dot-product'-like method to calculate the similarity of each post to the centroid. The higher the similarity, the more representative the post is of the cluster/persona.
    
    # get top n_posts most similar to centroid - based on the same cosine similarity
    top_local_idx = np.argsort(similarities)[::-1][:n_posts] # Argsort for the most similar posts within each of the clusters.
    top_global_idx = cluster_indices[top_local_idx]
    
    top_posts = df.iloc[top_global_idx][["title", "content"]].copy()
    top_posts["similarity"] = similarities[top_local_idx]
    
    print(f"  Total posts in cluster: {cluster_mask.sum():,}")
    print(f"  MININUM cos-similarity score: {similarities[top_local_idx[-1]]:.3f}")
    for title in top_posts["title"].head(5).tolist():
        print(f"    - {title[:80]}")
    
    # write to txt file - FOR STEPHEN'S MiroFish sim
    out_path = f"test_cluster_{k}_posts.txt"
    with open(out_path, "w") as f:
        for i, row in enumerate(top_posts.itertuples()):
            f.write(f"POST {i+1}:\n")
            f.write(f"Title: {row.title}\n")
            f.write(f"Content: {row.content}\n\n")

    time.sleep(3)
    
    print(f"  Saved {out_path}")